# Kedr: obshiy CHOD for INN from final_df

This notebook:
- reads `inn` from `final_df_june_2026_with_april_coef.csv`;
- gets `sum(nbi_ssp)` by `inn` from `Kedr.v_detail_dmkb`, `Kedr.v_detail_dmsb`, `Kedr.v_detail_dkb` for selected `yearmm`;
- calculates `obshiy_chod` as `max` across available source sums;
- adds overlap check where `src_cnt > 1`.

Business rule implemented:
- if an `inn` exists in one table, `obshiy_chod = sum(nbi_ssp)` from that table;
- if an `inn` exists in multiple tables, `obshiy_chod = max` of table-level sums.

In [ ]:
import os
import math
import re

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def chunked(values, size):
    if size <= 0:
        raise ValueError('chunk size must be positive')
    for i in range(0, len(values), size):
        yield values[i : i + size]


def in_sql_list(values):
    clean_values = [str(v).strip() for v in values if str(v).strip()]
    if not clean_values:
        return "''"
    return ', '.join([f"'{v}'" for v in clean_values])


def run_impala_fetch(imp, sql_text, mem_limit='8g'):
    with imp:
        imp.execute(f'set MEM_LIMIT={mem_limit}')
        return imp.fetch(sql_text)

In [ ]:
# Inputs / outputs
input_csv_path = '/home/jovyan/documents/Equaring/Data/final_df_june_2026_with_april_coef.csv'
output_csv_path = '/home/jovyan/documents/Equaring/Data/final_df_june_2026_with_april_coef_with_obshiy_chod.csv'
overlap_csv_path = '/home/jovyan/documents/Equaring/Data/final_df_june_2026_kedr_src_cnt_gt1.csv'

# Period in Kedr.v_detail_* (decimal yearmm)
target_yyyymm = 202605

# Query settings
chunk_size = 800
mem_limit = '8g'
save_outputs = True

In [ ]:
try:
    from connection_secrets import LAKE_USER, LAKE_PASSWORD
except Exception as exc:
    raise RuntimeError('Cannot import LAKE_USER/LAKE_PASSWORD from connection_secrets.py') from exc

keytab_path = '/home/jovyan/test_requests/tech.keytab'
kerberos_cfg = {
    'use_credentials': True,
    'update_keytab': False,
}

if os.path.exists(keytab_path):
    kerberos_cfg['keytab_path'] = keytab_path
    kerberos_cfg['update_keytab'] = True
else:
    print(f'WARNING: keytab not found: {keytab_path}')
    print('Using current ticket cache (kinit must be valid).')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos=kerberos_cfg,
    user_params={
        'user_name': LAKE_USER,
        'password': LAKE_PASSWORD,
    }
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
final_df = pd.read_csv(input_csv_path, dtype={'inn': 'string'})
if 'inn' not in final_df.columns:
    raise ValueError('Column `inn` not found in input CSV')

final_df['inn'] = final_df['inn'].map(normalize_inn)
final_df = final_df[final_df['inn'].notna()].copy()

inn_values = sorted(final_df['inn'].drop_duplicates().tolist())
if not inn_values:
    raise ValueError('No valid INN values found in input CSV')

print(f'rows in final_df after INN normalization: {len(final_df):,}')
print(f'unique INN to query in Kedr: {len(inn_values):,}')

In [ ]:
def build_kedr_sql(inn_scope, yyyymm):
    inn_sql = in_sql_list(inn_scope)
    return f"""
with src as (
    select inn, sum(nbi_ssp) as chod_value, 'dmkb' as src_name
    from Kedr.v_detail_dmkb
    where yearmm = {yyyymm}
      and inn in ({inn_sql})
    group by inn

    union all

    select inn, sum(nbi_ssp) as chod_value, 'dmsb' as src_name
    from Kedr.v_detail_dmsb
    where yearmm = {yyyymm}
      and inn in ({inn_sql})
    group by inn

    union all

    select inn, sum(nbi_ssp) as chod_value, 'dkb' as src_name
    from Kedr.v_detail_dkb
    where yearmm = {yyyymm}
      and inn in ({inn_sql})
    group by inn
)
select
    inn,
    max(case when src_name = 'dmkb' then chod_value end) as chod_dmkb,
    max(case when src_name = 'dmsb' then chod_value end) as chod_dmsb,
    max(case when src_name = 'dkb' then chod_value end) as chod_dkb,
    max(chod_value) as obshiy_chod,
    sum(case when src_name = 'dmkb' then 1 else 0 end)
      + sum(case when src_name = 'dmsb' then 1 else 0 end)
      + sum(case when src_name = 'dkb' then 1 else 0 end) as src_cnt
from src
group by inn
"""


parts = []
total_chunks = math.ceil(len(inn_values) / chunk_size)

for chunk_num, inn_scope in enumerate(chunked(inn_values, chunk_size), start=1):
    sql_text = build_kedr_sql(inn_scope, target_yyyymm)
    part_df = run_impala_fetch(imp, sql_text, mem_limit=mem_limit)
    parts.append(part_df)
    print(f'chunk {chunk_num}/{total_chunks}: {len(part_df):,} rows')

kedr_chod_df = (
    pd.concat(parts, ignore_index=True)
    if parts
    else pd.DataFrame(columns=['inn', 'chod_dmkb', 'chod_dmsb', 'chod_dkb', 'obshiy_chod', 'src_cnt'])
)

if not kedr_chod_df.empty:
    kedr_chod_df = (
        kedr_chod_df
        .groupby('inn', as_index=False)
        .agg({
            'chod_dmkb': 'max',
            'chod_dmsb': 'max',
            'chod_dkb': 'max',
            'obshiy_chod': 'max',
            'src_cnt': 'max',
        })
    )

print(f'kedr_chod_df rows: {len(kedr_chod_df):,}')
kedr_chod_df.head()

In [ ]:
inn_scope_df = final_df[['inn']].drop_duplicates().copy()

final_inn_chod_df = inn_scope_df.merge(
    kedr_chod_df[['inn', 'obshiy_chod', 'src_cnt']],
    on='inn',
    how='left'
)

src_cnt_gt1_df = (
    final_inn_chod_df[final_inn_chod_df['src_cnt'] > 1][['inn', 'obshiy_chod']]
    .sort_values(['obshiy_chod'], ascending=[False])
    .rename(columns={'inn': 'INN'})
    .reset_index(drop=True)
)

final_inn_chod_df = final_inn_chod_df[['inn', 'obshiy_chod']].rename(columns={'inn': 'INN'})

print(f'final_inn_chod_df rows: {len(final_inn_chod_df):,}')
print(f'unique INN in output: {final_inn_chod_df["INN"].nunique():,}')
print(f'filled obshiy_chod: {final_inn_chod_df["obshiy_chod"].notna().mean() * 100:.2f}%')
print(f'INN with src_cnt > 1: {len(src_cnt_gt1_df):,}')

print('\nOutput sample (INN + obshiy_chod):')
print(final_inn_chod_df.head(10))

print('\nOverlap sample (src_cnt > 1):')
print(src_cnt_gt1_df.head(50))

In [ ]:
if save_outputs:
    final_inn_chod_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    src_cnt_gt1_df.to_csv(overlap_csv_path, index=False, encoding='utf-8-sig')
    print(f'Saved INN + obshiy_chod file: {output_csv_path}')
    print(f'Saved overlap check file: {overlap_csv_path}')
else:
    print('save_outputs=False, files are not written')